In [1]:
import json
from pathlib import Path

In [ ]:
base_dir = Path(r'c:\Users\kbkav\OneDrive\Desktop\capstone')
from io import StringIO

import pandas as pd
from sklearn.model_selection import train_test_split
import unicodedata

output_dir = (base_dir / 'dataset' / 'final_dataset')
output_dir.mkdir(parents=True, exist_ok=True)

def try_read_csv(path, **kwargs):
    """Try reading a CSV with several encodings and fall back to replacement decoding."""
    encodings = ['utf-8', 'cp1252', 'latin-1']
    last_exc = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_exc = e
    # fallback: read file with replacement for undecodable bytes then parse
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()
    return pd.read_csv(StringIO(text), **kwargs)


MAX_ROWS_PER_SOURCE = 20000
RANDOM_STATE = 42


def normalize_text(*parts):
    cleaned = []
    for part in parts:
        if pd.isna(part):
            continue
        text = str(part).strip()
        if text and text.lower() != 'nan':
            cleaned.append(text)
    return ' '.join(cleaned).replace('\n', ' ').strip()


def sample_frame(frame, limit):
    if len(frame) <= limit:
        return frame.copy()
    return frame.sample(n=limit, random_state=RANDOM_STATE).reset_index(drop=True)


def build_source_frame(frame, text_columns, label_value, source_name, limit=MAX_ROWS_PER_SOURCE):
    frame = frame.copy()
    frame['text'] = frame[text_columns].apply(lambda row: normalize_text(*row.tolist()), axis=1)
    frame['label'] = label_value
    frame['source'] = source_name
    frame = frame[['text', 'label', 'source']]
    frame = frame[frame['text'].str.len() > 0]
    frame = sample_frame(frame, limit)
    return frame


def fix_mojibake(s):
    if not isinstance(s, str):
        return s
    try:
        decoded = s.encode('latin-1').decode('utf-8')
        if '�' not in decoded:
            return unicodedata.normalize('NFKC', decoded)
    except Exception:
        pass
    return unicodedata.normalize('NFKC', s)


def clean_text(s):
    """Normalize, attempt mojibake fix, remove Unicode control/format characters, collapse whitespace."""
    if not isinstance(s, str):
        return s
    # Normalize and attempt to repair mojibake
    s = fix_mojibake(s)
    s = unicodedata.normalize('NFKC', s)
    # Remove characters with Unicode category starting with 'C' (Other: control/format/private-use, surrogate, unassigned)
    cleaned_chars = []
    for ch in s:
        cat = unicodedata.category(ch)
        if cat.startswith('C'):
            # replace control/format with space
            cleaned_chars.append(' ')
        else:
            cleaned_chars.append(ch)
    s = ''.join(cleaned_chars)
    # Remove common replacement char if present
    s = s.replace('\ufffd', ' ')
    # Collapse whitespace
    s = ' '.join(s.split())
    return s.strip()


def load_liar_file(file_path, split_name, limit=MAX_ROWS_PER_SOURCE):
    liar_columns = [
        'id', 'label_raw', 'statement', 'subject', 'speaker', 'job_title',
        'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts',
        'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'
    ]
    liar = try_read_csv(file_path, sep='\t', header=None, names=liar_columns)
    mapping = {
        'true': 1,
        'mostly-true': 1,
        'half-true': 0,
        'false': 0,
        'barely-true': 0,
        'pants-fire': 0,
    }
    liar['label'] = liar['label_raw'].map(mapping)
    liar = liar.dropna(subset=['label'])
    liar['text'] = liar.apply(lambda row: normalize_text(row['statement'], row['subject'], row['context']), axis=1)
    liar['source'] = f'liar_{split_name}'
    liar['label'] = liar['label'].astype(int)
    liar = liar[['text', 'label', 'source']]
    liar = liar[liar['text'].str.len() > 0]
    return sample_frame(liar, limit)


frames = []
dataset_dir = base_dir / 'dataset'

archive_fake = try_read_csv(dataset_dir / 'archive' / 'Fake.csv', usecols=['title', 'text'])
frames.append(build_source_frame(archive_fake, ['title', 'text'], 0, 'archive_fake'))

archive_true = try_read_csv(dataset_dir / 'archive' / 'True.csv', usecols=['title', 'text'])
frames.append(build_source_frame(archive_true, ['title', 'text'], 1, 'archive_true'))

isot_fake = try_read_csv(dataset_dir / 'ISOT' / 'Fake.csv', usecols=['title', 'text'])
frames.append(build_source_frame(isot_fake, ['title', 'text'], 0, 'isot_fake'))

isot_true = try_read_csv(dataset_dir / 'ISOT' / 'True.csv', usecols=['title', 'text'])
frames.append(build_source_frame(isot_true, ['title', 'text'], 1, 'isot_true'))

gossipcop_fake = try_read_csv(dataset_dir / 'FakeNewsNet' / 'gossipcop_fake.csv', usecols=['title'])
frames.append(build_source_frame(gossipcop_fake, ['title'], 0, 'fakenewsnet_gossipcop_fake'))

gossipcop_real = try_read_csv(dataset_dir / 'FakeNewsNet' / 'gossipcop_real.csv', usecols=['title'])
frames.append(build_source_frame(gossipcop_real, ['title'], 1, 'fakenewsnet_gossipcop_real'))

politifact_fake = try_read_csv(dataset_dir / 'FakeNewsNet' / 'politifact_fake.csv', usecols=['title'])
frames.append(build_source_frame(politifact_fake, ['title'], 0, 'fakenewsnet_politifact_fake'))

politifact_real = try_read_csv(dataset_dir / 'FakeNewsNet' / 'politifact_real.csv', usecols=['title'])
frames.append(build_source_frame(politifact_real, ['title'], 1, 'fakenewsnet_politifact_real'))

buzzfeed = try_read_csv(dataset_dir / 'BuzzFeed' / 'facebook-fact-check.csv')
buzzfeed['Rating'] = buzzfeed['Rating'].astype(str).str.lower().str.strip()
buzzfeed = buzzfeed[buzzfeed['Rating'].isin(['mostly true', 'mostly false', 'mixture of true and false'])].copy()
buzzfeed['label'] = buzzfeed['Rating'].map({
    'mostly true': 1,
    'mostly false': 0,
    'mixture of true and false': 0,
})
buzzfeed['text'] = buzzfeed.apply(
    lambda row: normalize_text(row.get('Page'), row.get('Category'), row.get('Post Type'), row.get('Debate')),
    axis=1,
)
buzzfeed['source'] = 'buzzfeed'
buzzfeed = buzzfeed[['text', 'label', 'source']]
buzzfeed = buzzfeed[buzzfeed['text'].str.len() > 0]
frames.append(sample_frame(buzzfeed, MAX_ROWS_PER_SOURCE))

multifc = try_read_csv(dataset_dir / 'MultiFC' / 'merged_fakenews.csv', usecols=['text1', 'text2', 'label'])
multifc['label'] = pd.to_numeric(multifc['label'], errors='coerce')
multifc = multifc.dropna(subset=['label'])
multifc['label'] = multifc['label'].astype(int)
multifc['text'] = multifc.apply(lambda row: normalize_text(row['text1'], row['text2']), axis=1)
multifc['source'] = 'multifc'
multifc = multifc[['text', 'label', 'source']]
multifc = multifc[multifc['text'].str.len() > 0]
frames.append(sample_frame(multifc, MAX_ROWS_PER_SOURCE))

frames.append(load_liar_file(dataset_dir / 'LIAR Dataset' / 'train.tsv', 'train'))
frames.append(load_liar_file(dataset_dir / 'LIAR Dataset' / 'valid.tsv', 'valid'))
frames.append(load_liar_file(dataset_dir / 'LIAR Dataset' / 'test.tsv', 'test'))

final_dataset = pd.concat(frames, ignore_index=True)
# Normalize whitespace and ensure strings
final_dataset['text'] = final_dataset['text'].astype(str).str.replace(r'\\s+', ' ', regex=True).str.strip()

# Try to repair common mojibake and normalize unicode
final_dataset['text'] = final_dataset['text'].apply(fix_mojibake)

# Remove control/format characters and collapse whitespace
final_dataset['text'] = final_dataset['text'].apply(clean_text)
final_dataset = final_dataset[final_dataset['text'].ne('') & final_dataset['text'].notna()]
final_dataset = final_dataset.drop_duplicates(subset=['text', 'label']).reset_index(drop=True)

# BSDetector contains domain-level metadata rather than article text, so it is saved separately
# instead of being forced into the binary text classification dataset.
bsdetector_path = dataset_dir / 'BSDetector' / 'data.json'
bsdetector_output = output_dir / 'bsdetector_sources.csv'
bsdetector_data = json.loads(bsdetector_path.read_text(encoding='utf-8', errors='replace'))
bsdetector_frame = pd.DataFrame([
    {'domain': domain, 'type': values.get('type', ''), 'language': values.get('language', ''), 'notes': values.get('notes', '')}
    for domain, values in bsdetector_data.items()
])
bsdetector_frame.to_csv(bsdetector_output, index=False, encoding='utf-8')

label_counts = final_dataset['label'].value_counts().sort_index()
source_counts = final_dataset['source'].value_counts().sort_values(ascending=False)

final_dataset_path = output_dir / 'final_dataset.csv'
train_path = output_dir / 'train_split.csv'
test_path = output_dir / 'test_split.csv'
summary_path = output_dir / 'dataset_summary.json'

final_dataset.to_csv(final_dataset_path, index=False, encoding='utf-8')

train_df, test_df = train_test_split(
    final_dataset,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=final_dataset['label'],
)

train_df.to_csv(train_path, index=False, encoding='utf-8')
test_df.to_csv(test_path, index=False, encoding='utf-8')

summary = {
    'rows': int(len(final_dataset)),
    'label_counts': {str(k): int(v) for k, v in label_counts.items()},
    'source_counts': {str(k): int(v) for k, v in source_counts.items()},
    'final_dataset_path': str(final_dataset_path),
    'train_path': str(train_path),
    'test_path': str(test_path),
    'bsdetector_output': str(bsdetector_output),
}

summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Final dataset saved to: {final_dataset_path}')
print(f'Train split saved to: {train_path}')
print(f'Test split saved to: {test_path}')
print(f'BSDetector source inventory saved to: {bsdetector_output}')
print('Rows after merge and dedupe:', len(final_dataset))
print('Label distribution:')
print(label_counts.to_string())
print('Source distribution:')
print(source_counts.to_string())

Final dataset saved to: c:\Users\kbkav\OneDrive\Desktop\capstone\dataset\final_dataset\final_dataset.csv
Train split saved to: c:\Users\kbkav\OneDrive\Desktop\capstone\dataset\final_dataset\train_split.csv
Test split saved to: c:\Users\kbkav\OneDrive\Desktop\capstone\dataset\final_dataset\test_split.csv
BSDetector source inventory saved to: c:\Users\kbkav\OneDrive\Desktop\capstone\dataset\final_dataset\bsdetector_sources.csv
Rows after merge and dedupe: 90396
Label distribution:
label
0    38263
1    52133
Source distribution:
source
multifc                        20000
archive_true                   19805
fakenewsnet_gossipcop_real     15967
archive_fake                   15889
liar_train                     10235
fakenewsnet_gossipcop_fake      4896
liar_valid                      1283
liar_test                       1267
fakenewsnet_politifact_real      557
fakenewsnet_politifact_fake      427
buzzfeed                          70
